In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

C:\Users\muham\AppData\Local\Temp\ipykernel_17052\338479224.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    persist_directory="./intro-to-ds-lectures",
    embedding_function=embedding
)

d:\projects\RAG and Langchain\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2533.18it/s]
C:\Users\muham\AppData\Local\Temp\ipykernel_17052\2282194683.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [4]:
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,
        'lambda_mult': 0.7
    })

In [5]:
TEMPLATE = """
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
"""

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [15]:
chat = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=500
)

In [11]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        lecture = doc.metadata.get("Lecture Title", "")
        formatted.append(f"{doc.page_content}\n(Lecture: {lecture})")
    return "\n\n".join(formatted)

In [7]:
question = "What software do data scientists use?"

In [16]:
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | chat
    | StrOutputParser()
)

In [17]:
response = chain.invoke(question)

In [18]:
print(response)

Data scientists commonly use a mix of programming languages and specialized software tools. According to the provided context, the most popular tools are **R** and **Python**—both are versatile for data manipulation, statistical analysis, and end‑to‑end problem solving. In addition, many data scientists work with big‑data platforms such as **Apache Hadoop**, **Apache HBase**, and **MongoDB** to handle large volumes of data.

These tools are integrated into various data‑science platforms and are chosen for their flexibility and broad applicability across business and analytical tasks.

Resources: Lecture: Programming Languages & Software Employed in Data Science - All the Tools You Need
